In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
import scipy.stats as stats
from scipy.integrate import quad
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import utilities.plot_settings
from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D

def update(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)
    handle.set_markersize(3)

In [ ]:
# Characteristic white dwarf radius in [cm].
R = 9.e8 #7.e8

# Characteristic white dwarf mass in solar masses.
M = 0.6 * const.M_SUN

# Dimensionless coefficients k_0, k_1, k_2 for a force-free magnetosphere
# taken from Spitkovsky (2006) and Philippov et al. (2014).
# For comparison, in vacuum k_0 = 0 and k_1 = k_2 = 2/3.
k_coefficients = [1.0, 1.0, 1.0]

# Canonical white dwarf moment of inertia in [g cm^2] assuming a perfect solid sphere.
I = 2.0 / 5.0 * M * R ** 2

# Auxiliary quantity beta as defined in eq. (72) of Pons & Vigano (2019).
beta = 1./4. * R ** 6 / (I * const.C ** 3)
#beta = np.pi ** 2 * NS_radius ** 6 / (NS_inertia * const.c ** 3)
print(beta)

# Assume an inclination angle in [rad].
chi = 0.

beta_1 = beta * (k_coefficients[0] + k_coefficients[1] * np.sin(chi) ** 2)
print(beta_1)

In [ ]:
def B_from_timing(P: float, Pdot: float) -> float:
    """
    B field estimated from timing properties. 
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Pdot (float): period derivative of a simulated pulsar in [s/s].   

    Returns:
        (float): value of the dipolar component of the magnetic field at the
        magnetic pole for a simulated neutron star, measured in [G].
    """

    # Period derivative.
    B = np.sqrt(P * Pdot / (4 * np.pi**2 * beta_1))

    return B

def Edot_from_timing(P: float, Pdot: float) -> float:
    """
    Pulsar rotational power loss from timing properties. 
    
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Pdot (float): period derivative of a simulated pulsar in [s/s].   

    Returns:
        (float): characteristic age in [yr].     
    """

    # Period derivative.
    Erot_dot = (2.*np.pi)**2 * I * Pdot / P**3

    return Erot_dot

In [ ]:
data_i = pd.read_pickle(
    "../toremove/nanda_experiment/WD_exp4_1e7/initial_population.pkl.gz",
    compression="gzip",
)
data_i.head()

In [ ]:
data_f = pd.read_pickle(
    "../toremove/nanda_experiment/WD_exp4_1e7/final_population.pkl.gz",
    compression="gzip",
)
data_f.head()

In [ ]:
P_i = data_i["P"]["[s]"].to_numpy()
P_dot_i = data_i["P_dot"]["[s s^-1]"].to_numpy()

P_f = data_f["P"]["[s]"].to_numpy()
P_dot_f = data_f["P_dot"]["[s s^-1]"].to_numpy()

Erot = ((2*np.pi)**2)*I*(P_dot_f/P_f**3)

intercept_radio = data_f[('intercepted_radio',' ')] == 1

In [ ]:
P_min = 1.e-2
P_max = 1.e10
Pdot_min = 1.e-32
Pdot_max = 1.e-4

log_P_edges = np.linspace(np.log10(P_min), np.log10(P_max), 71)
log_P_centers = 0.5 * (log_P_edges[1:] + log_P_edges[:-1])
P_edges = 10**log_P_edges
P_centers = 10**log_P_centers

log_Pdot_edges = np.linspace(np.log10(Pdot_min), np.log10(Pdot_max), 71)
log_Pdot_centers = 0.5 * (log_Pdot_edges[1:] + log_Pdot_edges[:-1])
Pdot_edges = 10**log_Pdot_edges
Pdot_centers = 10**log_Pdot_centers

P_grid, Pdot_grid = np.meshgrid(P_centers, Pdot_centers, indexing='ij')

B_timing = B_from_timing(P_grid, Pdot_grid)
Edot_timing = Edot_from_timing(P_grid, Pdot_grid)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [1, 3], "hspace": 0})

plt.xscale('log') 
plt.yscale('log') 

# Plot the distribution of P_i at the top
P_bins = np.logspace(-2, 10, 31)

ax1.set_ylabel("# WDs")
ax1.set_xscale('log') 
ax1.set_yscale('log') 
ax1.tick_params(axis="x", labelbottom=False)
ax1.set_yticks([1e2, 1e4, 1e6])

ax1.hist(
    P_i, 
    bins=P_bins,
    histtype="step",
    linewidth=4,
    color="grey",
    #density=True,
)
ax1.hist(
    P_f, 
    bins=P_bins,
    histtype="step",
    linewidth=4,
    color="tab:pink",
    #density=True,
)
ax1.hist(
    P_f[intercept_radio], 
    bins=P_bins,
    histtype="step",
    linewidth=4,
    color="indigo",
    #density=True,
)
ax1.text(1e8, 1e5, "WD1", fontsize=35)

ax2.set_xlabel(r"$P$ [s]")
ax2.set_ylabel(r"$\dot{P}$")
ax2.set_xscale('log') 
ax2.set_yscale('log') 
ax2.set_xlim(P_min,P_max)
ax2.set_ylim(Pdot_min,Pdot_max)

contour_B = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(B_timing), 
    levels = np.array([2.,4.,6.,8.,10.,12.]), 
    colors='black',
    linestyles='dashed',
    alpha=0.7
    #interpolation='none'
)
fmt = {}
strs = ['$10^{2}$ G', '$10^{4}$ G', '$10^{6}$ G', '$10^{8}$ G', '$10^{10}$ G', '$10^{12}$ G']
for l,s in zip( contour_B.levels, strs ):
    fmt[l] = rf"{s}"
manual_locations = [(5.e-2, 1e-22), (5.e-2, 1e-19), (5.e-2, 1e-15), (5.e-2, 1e-10), (5.e-2, 1e-7), (10., 1e-6)]
ax2.clabel(
    contour_B, 
    contour_B.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20, 
    colors='black'
)

contour_Edot = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(Edot_timing),
    levels = np.array([5.,15.,25.,35.]), 
    colors='black',
    linestyles='dashed',
    alpha=0.7
    #interpolation='none'
)
fmt = {}
strs = ['$10^{5}$ erg s$^{-1}$', '$10^{15}$ erg s$^{-1}$', '$10^{25}$ erg s$^{-1}$', '$10^{35}$ erg s$^{-1}$']
for l,s in zip( contour_Edot.levels, strs ):
    fmt[l] = rf"{s}"
manual_locations = [(1e9, 1e-20), (1e9, 1e-12), (1e7, 1e-7), (1e3, 1e-7)]
ax2.clabel(
    contour_Edot, 
    contour_Edot.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20, 
    colors='black'
)

ax2.plot(
    P_i,
    P_dot_i,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=3,
    alpha=0.1,
    rasterized=True,
    label = 'Initial population'
)

ax2.plot(
    P_f,
    P_dot_f,
    linestyle="None",
    marker="o",
    color="tab:pink",
    markersize=3,
    alpha=0.1,
    rasterized=True,
    label = 'Final population'
)
ax2.plot(
    P_f[intercept_radio],
    P_dot_f[intercept_radio],
    linestyle="None",
    marker="o",
    #mfc='none',
    color="indigo",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los'
)

#plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20,markerscale=5)
plt.legend(frameon=False, loc=3, fontsize=25, markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})

plt.savefig("fig4_WD4_ppdot.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    P_f,
    Erot,
    linestyle="None",
    marker="o",
    color="tab:pink",
    markersize=3,
    alpha=0.1,
    rasterized=True,
    label = 'Final population'
)

ax.plot(
    P_f[intercept_radio],
    Erot[intercept_radio],
    linestyle="None",
    marker="o",
    color="indigo",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercepting our los'
)
ax.text(1e8, 1e35, "WD1", fontsize=35)

#plt.axvline(10**(-0.7))
ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$\dot{E}_{\rm rot}$")
ax.set_xlim(P_min,P_max)
plt.xscale('log') 
plt.yscale('log') 

#plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20,markerscale=5)
plt.legend(frameon=False, loc="lower left", fontsize=25,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})

plt.grid()
plt.savefig("fig4_WD4_Erot.pdf")
plt.show()